# Vidu Q3 Turbo Reference-to-Video API 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台调用 **Vidu Q3 Turbo (Reference-to-Video)** 模型 API。

## 模型简介

Vidu Q3 Turbo Reference-to-Video 是基于参考图片生成视频的模型。你可以传入 1-7 张参考图片，并用 prompt 描述希望生成的视频内容、动作和风格。

- **参考生视频**：通过 `reference_image_urls` 提供角色、物体、场景或风格参考
- **Q3 能力**：`prompt` 最多 2000 字符，`duration` 支持 3-16 秒
- **分辨率控制**：支持 `1080p`、`720p`、`560p`
- **画幅控制**：支持 `16:9`、`9:16`、`3:4`、`4:3`、`1:1`

Turbo 版本生成速度更快，适合需要快速出片的参考生视频场景。

**API 端点**：`fal-ai/vidu/q3/reference-to-video/turbo`

## 1. 环境准备

首先安装 `fal-client` 依赖包，并配置 API Key 和七牛 AIToken 队列服务地址。

In [ ]:
# 安装 fal-client
%pip install fal-client -q

In [ ]:
import os

# 设置七牛 AIToken 队列服务地址
os.environ["FAL_QUEUE_RUN_HOST"] = "api.qnaigc.com/queue"

# 设置 FAL_KEY 环境变量
# 也可以在终端中通过 export QINIU_API_KEY="xxx" 设置
api_key = os.environ.get("QINIU_API_KEY")
if not api_key:
    raise ValueError("环境变量 QINIU_API_KEY 未设置")
os.environ["FAL_KEY"] = api_key

In [ ]:
import time
import fal_client
from IPython.display import Video, display

## 2. 基础用法 - 队列调用

七牛 AIToken 平台使用队列模式调用 API：先提交请求（`submit`），然后轮询状态（`status`），最后获取结果（`result`）。

下面先封装一个通用的调用辅助函数，后续示例都会复用。

In [ ]:
def run_fal_task(endpoint, arguments, poll_interval=5):
    """提交任务并轮询等待结果返回"""
    handler = fal_client.submit(endpoint, arguments=arguments)
    request_id = handler.request_id
    print(f"请求已提交，request_id: {request_id}")

    while True:
        status = fal_client.status(endpoint, request_id, with_logs=True)
        print(f"当前状态: {type(status).__name__}")

        if isinstance(status, fal_client.Completed):
            print("任务完成!")
            break

        if hasattr(status, "logs") and status.logs:
            for log in status.logs:
                print(log["message"])

        time.sleep(poll_interval)

    return fal_client.result(endpoint, request_id)


result = run_fal_task(
    "fal-ai/vidu/q3/reference-to-video/turbo",
    arguments={
        "prompt": "A character walking through a beach catching an apple.",
        "reference_image_urls": [
        "https://storage.googleapis.com/falserverless/web-examples/vidu/new-examples/reference1.png",
        "https://storage.googleapis.com/falserverless/web-examples/vidu/new-examples/reference2.png",
        "https://storage.googleapis.com/falserverless/web-examples/vidu/new-examples/reference3.png"
        ],
        "aspect_ratio": "16:9",
        "duration": 5,
        "resolution": "1080p",
    },
)

print(result)

In [ ]:
video_url = result["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 3. 参数说明与高级用法

### 可配置参数一览

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `prompt` | string | *必填* | 文本提示词，描述视频内容、动作和风格，最多 2000 字符 |
| `reference_image_urls` | array | *必填* | 参考图片 URL 列表，支持 1-7 张 |
| `aspect_ratio` | string | *必填* | 视频宽高比：`16:9`、`9:16`、`3:4`、`4:3`、`1:1` |
| `duration` | integer | `5` | 视频时长，Q3 参考生视频支持 3-16 秒 |
| `resolution` | string | `1080p` | 分辨率：`1080p`、`720p`、`560p` |
| `seed` | integer | 随机 | 随机种子，便于复现相近结果 |
| `movement_amplitude` | string | `auto` | 运动幅度：`auto`、`small`、`medium`、`large` |
| `bgm` | boolean | 可选 | 是否添加背景音乐 |
| `audio` | boolean | 可选 | 是否开启音视频直出 |

> `reference_video_urls` 仅适用于 `viduq2-pro`，不要在 Q3 / Q3 Turbo 参考生视频接口中传入。

In [ ]:
result_advanced = run_fal_task(
    "fal-ai/vidu/q3/reference-to-video/turbo",
    arguments={
        "prompt": "A character walking through a beach catching an apple.",
        "reference_image_urls": [
        "https://storage.googleapis.com/falserverless/web-examples/vidu/new-examples/reference1.png",
        "https://storage.googleapis.com/falserverless/web-examples/vidu/new-examples/reference2.png",
        "https://storage.googleapis.com/falserverless/web-examples/vidu/new-examples/reference3.png"
        ],
        "aspect_ratio": "9:16",
        "duration": 8,
        "resolution": "720p",
        "movement_amplitude": "medium",
        "seed": 42,
        "bgm": True,
    },
)

video_url = result_advanced["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 4. 队列模式 - 手动分步调用

如果你需要在提交任务后做其他处理，稍后再查询结果，可以手动拆分为提交、查询和取结果三个步骤。

In [ ]:
handler = fal_client.submit(
    "fal-ai/vidu/q3/reference-to-video/turbo",
    arguments={
        "prompt": "A character walking through a beach catching an apple.",
        "reference_image_urls": [
        "https://storage.googleapis.com/falserverless/web-examples/vidu/new-examples/reference1.png",
        "https://storage.googleapis.com/falserverless/web-examples/vidu/new-examples/reference2.png",
        "https://storage.googleapis.com/falserverless/web-examples/vidu/new-examples/reference3.png"
        ],
        "aspect_ratio": "16:9",
        "duration": 5,
        "resolution": "1080p",
    },
)

request_id = handler.request_id
print(f"请求已提交，request_id: {request_id}")

In [ ]:
while True:
    status = fal_client.status(
        "fal-ai/vidu/q3/reference-to-video/turbo",
        request_id,
        with_logs=True,
    )
    print(f"当前状态: {type(status).__name__}")

    if isinstance(status, fal_client.Completed):
        print("任务完成!")
        break

    time.sleep(5)

In [ ]:
result_async = fal_client.result(
    "fal-ai/vidu/q3/reference-to-video/turbo",
    request_id,
)

video_url = result_async["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 5. Schema 参考

### Input Schema

```json
{
  "prompt": "string (必填, 最多 2000 字符)",
  "reference_image_urls": ["string (必填, 图片 URL, 1-7 张)"],
  "aspect_ratio": "16:9 | 9:16 | 3:4 | 4:3 | 1:1 (必填)",
  "duration": "integer (可选, 默认 5, 范围 3-16)",
  "resolution": "1080p | 720p | 560p (可选, 默认 1080p)",
  "seed": "integer (可选, 默认随机)",
  "movement_amplitude": "auto | small | medium | large (可选, 默认 auto)",
  "bgm": "boolean (可选)",
  "audio": "boolean (可选)"
}
```

### Output Schema

```json
{
  "video": {
    "url": "string (视频文件 URL)",
    "duration": "integer (视频时长, 秒)",
    "content_type": "string (视频 MIME 类型, 例如 video/mp4)"
  }
}
```

### 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [fal-client Python 文档](https://docs.fal.ai/reference/client-libraries/python/fal_client)
- [七牛 API 调用示例文档](https://apidocs.qnaigc.com)